# Boundary Unlearning · CIFAR-100

Attach [alincijov/cifar-100](https://www.kaggle.com/datasets/alincijov/cifar-100)
with **Add Input**, enable Internet for the GitHub code download, and select a GPU.
The dataset contains `train`, `test` and `meta` pickle files; no dataset download is needed.

This experiment uses **100 fine classes**, not the 20 coarse superclasses.
Both training and FGSM use mean/std `(0.5, 0.5, 0.5)` by default (`matched`).
AllCNN has 100 outputs; Boundary Expanding temporarily adds output 101.

The `paper` profile inherits the CIFAR-10 baseline settings (30 training epochs,
LR 0.1, batch 64; 10 unlearning epochs, LR 1e-5). These are initial settings for
this CIFAR-100 experiment, not validated CIFAR-100 results. Forgetting one fine
class uses 500 training images and approximately 80 updates with these settings.

Outputs go to `/kaggle/working/result/`, with `cifar100_` in run-specific filenames.
The final **Export results** cell creates `result.zip` and a download link.


In [2]:
# Run this cell first, both locally and on Kaggle.
from pathlib import Path
import importlib
import subprocess
import sys

GITHUB_REPO = "https://github.com/ngvihoa/boundary-unlearning.git"  
GITHUB_REF = "main"  # Branch, tag, or commit SHA. Use a SHA to repeat an experiment.
REPO_DIR = Path("/kaggle/working/boundary-unlearning")

if GITHUB_REPO:
    if not (REPO_DIR / ".git").exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--no-checkout",
            GITHUB_REPO, str(REPO_DIR),
        ], check=True)
    subprocess.run([
        "git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", GITHUB_REF,
    ], check=True)
    subprocess.run([
        "git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD",
    ], check=True)
    CODE_ROOT = REPO_DIR
else:
    CODE_ROOT = Path.cwd()

if not (CODE_ROOT / "boundary_unlearning" / "__init__.py").is_file():
    raise FileNotFoundError(
        "Set GITHUB_REPO to your GitHub HTTPS URL on Kaggle, "
        "or run this notebook from the local repository root."
    )

sys.path.insert(0, str(CODE_ROOT))
importlib.invalidate_caches()
print("Code root:", CODE_ROOT)
if (CODE_ROOT / ".git").exists():
    print("Code commit:", subprocess.check_output(
        ["git", "-C", str(CODE_ROOT), "rev-parse", "HEAD"], text=True,
    ).strip())
# For small code changes, reload the affected module and rerun its caller cells only.


Code root: /kaggle/working/boundary-unlearning
Code commit: 17e45f710fcb122d7564a7325102b907f1261b1d


From https://github.com/ngvihoa/boundary-unlearning
 * branch            main       -> FETCH_HEAD
HEAD is now at 17e45f7 add documents


In [ ]:
import csv
import hashlib
import json
import platform
import random
from pathlib import Path

import pandas as pd
import numpy as np
import torch
import torchvision
from torch.utils.data import Subset
from functools import partial
from IPython.display import display

from boundary_unlearning import cifar100, data, models, training, unlearning
from boundary_unlearning.reporting import results_table

# ============================================================
# 1. EXPERIMENT CONFIGURATION
# ============================================================
DATASET_NAME = "CIFAR-100"
DATASET_SLUG = "alincijov/cifar-100"
LABEL_TYPE = "fine"
RUN_PROFILE = "paper"  # "smoke", "paper", "repo_cli"

METHODS_TO_RUN = [
    "boundary_shrink",
    "boundary_expanding",
    "boundary_shrink_weight_assign",
    "boundary_shrink_weight_assign_distill",
    "boundary_shrink_weight_assign_boundary_distill",
]

FORGET_CLASS = 4
SEED = 2022
BATCH_SIZE = 64
NUM_WORKERS = 2

REUSE_CHECKPOINTS = False
ATTACK_NORMALIZATION = "matched"  # Use the same normalization as the CIFAR-100 input.

FGSM_BOUND = 0.1
UNLEARN_LR = 1e-5
MOMENTUM = 0.9
NUM_CLASSES = 100
EXPECTED_TRAIN_IMAGES = 50_000
EXPECTED_TEST_IMAGES = 10_000
# Optional Enhanced Boundary Shrink, matching the author's defaults.
EXTRA_EXP = None  # None, "curv", or "weight_assign"
CURVATURE_LAMBDA = 0.7
REWEIGHT_BIAS = -0.5
REWEIGHT_SLOPE = 5.0
# Variants from no-refactor-new.ipynb; independent of EXTRA_EXP above.
WEIGHT_EPSILON_STEP = 1 / 255
RETAIN_KL_LAMBDA = 0.1
RETAIN_KL_TEMPERATURE = 2.0
# Distance search always uses input mean/std (matched), as in that notebook.

# ============================================================
# 2. TRAINING CONFIGURATION
# ============================================================
if RUN_PROFILE == "smoke":
    ORIGINAL_EPOCHS = 1
    ORIGINAL_LR = 0.1
    UNLEARN_EPOCHS = 1
    MAX_BATCHES_PER_EPOCH = 3
    EVALUATE_TRAIN_SPLITS = False

elif RUN_PROFILE == "paper":
    ORIGINAL_EPOCHS = 30
    ORIGINAL_LR = 0.1
    UNLEARN_EPOCHS = 10
    MAX_BATCHES_PER_EPOCH = None
    EVALUATE_TRAIN_SPLITS = True

elif RUN_PROFILE == "repo_cli":
    ORIGINAL_EPOCHS = 50
    ORIGINAL_LR = 0.01
    UNLEARN_EPOCHS = 10
    MAX_BATCHES_PER_EPOCH = None
    EVALUATE_TRAIN_SPLITS = True

else:
    raise ValueError(
        "RUN_PROFILE must be smoke, paper, or repo_cli"
    )

if ATTACK_NORMALIZATION not in {"matched", "repo_exact"}:
    raise ValueError("Unsupported ATTACK_NORMALIZATION")

if EXTRA_EXP not in {None, "curv", "weight_assign"}:
    raise ValueError("Unsupported EXTRA_EXP")

if not 0 <= FORGET_CLASS < NUM_CLASSES:
    raise ValueError(f"FORGET_CLASS must be between 0 and {NUM_CLASSES - 1}")

unlearning.validate_methods(METHODS_TO_RUN, extra_exp=EXTRA_EXP)

# ============================================================
# 3. KAGGLE PATHS
# ============================================================
INPUT_ROOT = Path("/kaggle/input")
DATA_ROOT = Path(
    "/kaggle/input/datasets/alincijov/cifar-100"
)

# Kaggle also exposes older dataset mounts without the owner path.
if not DATA_ROOT.is_dir() and Path("/kaggle/input/cifar-100").is_dir():
    DATA_ROOT = Path("/kaggle/input/cifar-100")

OUTPUT_ROOT = Path("/kaggle/working")
RESULT_ROOT = OUTPUT_ROOT / "result"
CHECKPOINT_ROOT = RESULT_ROOT / "checkpoints"

# Kaggle Input is read-only; create directories only in working.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

# ============================================================
# 4. NORMALIZATION
# ============================================================
CIFAR_MEAN = (0.5, 0.5, 0.5)
CIFAR_STD = (0.5, 0.5, 0.5)

# Used only if ATTACK_NORMALIZATION is changed to "repo_exact".
REPO_ATTACK_MEAN = (0.4914, 0.4822, 0.2265)
REPO_ATTACK_STD = (0.2023, 0.1994, 0.2010)

# ============================================================
# 5. REPRODUCIBILITY
# ============================================================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ============================================================
# 6. ENVIRONMENT CHECK
# ============================================================
print("=" * 60)
print("ENVIRONMENT INFORMATION")
print("=" * 60)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("NumPy:", np.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is unavailable. Enable GPU in Kaggle settings.")

print("\nInput directories:")
for path in INPUT_ROOT.iterdir():
    print(" -", path)

print("\nData root exists:", DATA_ROOT.exists())
print("Output root:", OUTPUT_ROOT)
print("Checkpoint root:", CHECKPOINT_ROOT)
print("Result root:", RESULT_ROOT)
print("\nConfiguration initialized successfully.")


In [4]:
# Load the attached Kaggle pickles, including the labeled test split.
DATA_ROOT = cifar100.resolve_data_root(DATA_ROOT)
train_dataset, evaluation_dataset = cifar100.load_cifar100(
    DATA_ROOT, mean=CIFAR_MEAN, std=CIFAR_STD,
)
CIFAR_CLASSES = train_dataset.classes
train_labels = train_dataset.targets
test_labels = evaluation_dataset.targets

if len(train_dataset) != EXPECTED_TRAIN_IMAGES or len(evaluation_dataset) != EXPECTED_TEST_IMAGES:
    raise ValueError("Expected the complete CIFAR-100 train/test splits (50,000 / 10,000)")
train_class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
test_class_counts = np.bincount(test_labels, minlength=NUM_CLASSES)
if not np.all(train_class_counts == EXPECTED_TRAIN_IMAGES // NUM_CLASSES):
    raise ValueError("Training fine-label counts do not match the balanced CIFAR-100 split")
if not np.all(test_class_counts == EXPECTED_TEST_IMAGES // NUM_CLASSES):
    raise ValueError("Test fine-label counts do not match the balanced CIFAR-100 split")

display(pd.DataFrame([
    {"Split": "Train", "Images": len(train_dataset), "Classes": NUM_CLASSES,
     "Images per class": int(train_class_counts[FORGET_CLASS])},
    {"Split": "Test", "Images": len(evaluation_dataset), "Classes": NUM_CLASSES,
     "Images per class": int(test_class_counts[FORGET_CLASS])},
]).set_index("Split"))
print(f"Forget class: {FORGET_CLASS} ({CIFAR_CLASSES[FORGET_CLASS]})")


,Images,Classes,Images per class
Split,,,
Train,50000,100,500
Test,10000,100,100


Forget class: 4 (beaver)


In [5]:
train_fit_set = train_dataset
train_forget_indices = np.flatnonzero(train_labels == FORGET_CLASS).tolist()
train_retain_indices = np.flatnonzero(train_labels != FORGET_CLASS).tolist()
test_labels = np.asarray(evaluation_dataset.targets, dtype=np.int64)
test_forget_indices = np.flatnonzero(test_labels == FORGET_CLASS).tolist()
test_retain_indices = np.flatnonzero(test_labels != FORGET_CLASS).tolist()

train_forget_set = Subset(train_dataset, train_forget_indices)
train_retain_set = Subset(train_dataset, train_retain_indices)
test_forget_set = Subset(evaluation_dataset, test_forget_indices)
test_retain_set = Subset(evaluation_dataset, test_retain_indices)

print("Train fit:", len(train_fit_set))
print("Train forget:", len(train_forget_set))
print("Train retain:", len(train_retain_set))
print("Official test:", len(evaluation_dataset))
print("Official test forget:", len(test_forget_set))
print("Official test retain:", len(test_retain_set))


Train fit: 50000
Train forget: 500
Train retain: 49500
Official test: 10000
Official test forget: 100
Official test retain: 9900


In [6]:
PIN_MEMORY = DEVICE.type == "cuda"
make_loader = partial(
    data.make_loader, batch_size=BATCH_SIZE, seed=SEED,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
)

train_all_loader = make_loader(train_fit_set, shuffle=True)
train_retain_loader = make_loader(train_retain_set, shuffle=True)
train_retain_eval_loader = make_loader(train_retain_set, shuffle=False)
train_forget_eval_loader = make_loader(train_forget_set, shuffle=False)

test_loader = make_loader(evaluation_dataset, shuffle=False)
test_forget_loader = make_loader(test_forget_set, shuffle=False)
test_retain_loader = make_loader(test_retain_set, shuffle=False)

class_count = np.bincount(train_labels, minlength=NUM_CLASSES)

print(f"Fine classes: {len(CIFAR_CLASSES)} | Forget samples per epoch: {len(train_forget_set)}")
print("Train all batches:", len(train_all_loader))
print("Train retain batches:", len(train_retain_loader))
print("Train retain evaluation batches:", len(train_retain_eval_loader))
print("Train forget evaluation batches:", len(train_forget_eval_loader))
print("Official test batches:", len(test_loader))
print("Official test forget batches:", len(test_forget_loader))
print("Official test retain batches:", len(test_retain_loader))

forget_batches = len(make_loader(train_forget_set))
if MAX_BATCHES_PER_EPOCH is not None:
    forget_batches = min(forget_batches, MAX_BATCHES_PER_EPOCH)
print("Unlearning steps per method:", UNLEARN_EPOCHS * forget_batches)


Fine classes: 100 | Forget samples per epoch: 500
Train all batches: 782
Train retain batches: 774
Train retain evaluation batches: 774
Train forget evaluation batches: 8
Official test batches: 157
Official test forget batches: 2
Official test retain batches: 155
Unlearning steps per method: 80


In [7]:
# ============================================================
# PHASE 3: MODEL ARCHITECTURE
# ============================================================

shape_model = models.AllCNN(num_classes=NUM_CLASSES).to(DEVICE)
shape_model.apply(models.init_linear_like_repo)
shape_model.eval()

with torch.no_grad():
    sample_images = torch.randn(2, 3, 32, 32, device=DEVICE)
    sample_logits = shape_model(sample_images)

print("Device:", DEVICE)
print("Model:", shape_model.__class__.__name__)
print("Sample input shape:", sample_images.shape)
print("Sample logits shape:", sample_logits.shape)
print("Total parameters:", sum(p.numel() for p in shape_model.parameters()))
assert sample_logits.shape == (2, NUM_CLASSES)

del shape_model, sample_images, sample_logits


Device: cuda
Model: AllCNN
Sample input shape: torch.Size([2, 3, 32, 32])
Sample logits shape: torch.Size([2, 100])
Total parameters: 415540


In [8]:
create_or_load_model = partial(
    training.create_or_load_model, device=DEVICE,
    reuse_checkpoints=REUSE_CHECKPOINTS, num_classes=NUM_CLASSES,
    epochs=ORIGINAL_EPOCHS, learning_rate=ORIGINAL_LR, momentum=MOMENTUM,
    max_batches_per_epoch=MAX_BATCHES_PER_EPOCH,
)


In [ ]:
baseline_tag_settings = {
    "dataset": DATASET_NAME,
    "dataset_slug": DATASET_SLUG,
    "num_classes": NUM_CLASSES,
    "label_type": LABEL_TYPE,
    "input_mean": CIFAR_MEAN,
    "input_std": CIFAR_STD,
    "attack_mean": CIFAR_MEAN if ATTACK_NORMALIZATION == "matched" else REPO_ATTACK_MEAN,
    "attack_std": CIFAR_STD if ATTACK_NORMALIZATION == "matched" else REPO_ATTACK_STD,
    "profile": RUN_PROFILE,
    "data_root": str(DATA_ROOT),
    "implementation_version": "cifar100_v1",
    "evaluation_dataset": "kaggle_cifar100_test_fine",
    "checkpoint_selection": "best_test_accuracy",
    "extra_exp": EXTRA_EXP,
    "curvature_lambda": CURVATURE_LAMBDA,
    "reweight_bias": REWEIGHT_BIAS,
    "reweight_slope": REWEIGHT_SLOPE,
    "method": ["boundary_shrink", "boundary_expanding"],
    "forget_class": FORGET_CLASS,
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "original_epochs": ORIGINAL_EPOCHS,
    "original_lr": ORIGINAL_LR,
    "unlearn_epochs": UNLEARN_EPOCHS,
    "unlearn_lr": UNLEARN_LR,
    "momentum": MOMENTUM,
    "fgsm_bound": FGSM_BOUND,
    "attack_normalization": ATTACK_NORMALIZATION,
    "max_batches_per_epoch": MAX_BATCHES_PER_EPOCH,
}

baseline_hash = hashlib.sha256(
    json.dumps(baseline_tag_settings, sort_keys=True).encode("utf-8")
).hexdigest()[:8]

baseline_tag = f"cifar100_{RUN_PROFILE}_forget{FORGET_CLASS}_{baseline_hash}"

original_checkpoint = CHECKPOINT_ROOT / f"original_{baseline_tag}.pt"
retrain_checkpoint = CHECKPOINT_ROOT / f"retrain_{baseline_tag}.pt"

print("Baseline tag:", baseline_tag)
print("Original checkpoint:", original_checkpoint)
print("Retrain checkpoint:", retrain_checkpoint)

# New experiment identity includes all added settings, without renaming baselines.
tag_settings = {
    **baseline_tag_settings,
    "method": METHODS_TO_RUN,
    "distance_implementation_version": "boundary_distance_v1",
    "epsilon_step": WEIGHT_EPSILON_STEP,
    "distance_normalization": "matched",
    "retain_kl_lambda": RETAIN_KL_LAMBDA,
    "retain_kl_temperature": RETAIN_KL_TEMPERATURE,
    "boundary_kl_initializer": "boundary_shrink_weight_assign",
    "boundary_kl_teacher": "boundary_shrink",
    "input_mean": CIFAR_MEAN,
    "input_std": CIFAR_STD,
}
tag_hash = hashlib.sha256(json.dumps(tag_settings, sort_keys=True).encode("utf-8")).hexdigest()[:8]
run_tag = f"cifar100_{RUN_PROFILE}_forget{FORGET_CLASS}_{tag_hash}"
print("Comparison run tag:", run_tag)
# This cell only prepares paths/configuration. It does not train any models.


In [ ]:
original_model, original_train_seconds = create_or_load_model(
    "original",
    train_all_loader,
    original_checkpoint,
    test_loader,
)


In [10]:
retrain_model, retrain_train_seconds = create_or_load_model(
    "retrain",
    train_retain_loader,
    retrain_checkpoint,
    test_retain_loader,
)

print("Original model ready:", original_model is not None)
print("Retrain model ready:", retrain_model is not None)
print("Original training time:", original_train_seconds)
print("Retrain training time:", retrain_train_seconds)


[retrain] Epoch 1/30 | Loss: 3.9488 | Accuracy: 0.0830 | Test: 0.1211
[retrain] Epoch 2/30 | Loss: 3.3344 | Accuracy: 0.1808 | Test: 0.1912
[retrain] Epoch 3/30 | Loss: 2.7757 | Accuracy: 0.2847 | Test: 0.3107
[retrain] Epoch 4/30 | Loss: 2.3651 | Accuracy: 0.3679 | Test: 0.3676
[retrain] Epoch 5/30 | Loss: 2.0840 | Accuracy: 0.4308 | Test: 0.4397
[retrain] Epoch 6/30 | Loss: 1.8697 | Accuracy: 0.4826 | Test: 0.4483
[retrain] Epoch 7/30 | Loss: 1.6915 | Accuracy: 0.5242 | Test: 0.4722
[retrain] Epoch 8/30 | Loss: 1.5528 | Accuracy: 0.5565 | Test: 0.4621
[retrain] Epoch 9/30 | Loss: 1.4248 | Accuracy: 0.5872 | Test: 0.5118
[retrain] Epoch 10/30 | Loss: 1.3086 | Accuracy: 0.6188 | Test: 0.5114
[retrain] Epoch 11/30 | Loss: 1.1996 | Accuracy: 0.6442 | Test: 0.5116
[retrain] Epoch 12/30 | Loss: 1.1131 | Accuracy: 0.6663 | Test: 0.5060
[retrain] Epoch 13/30 | Loss: 1.0152 | Accuracy: 0.6927 | Test: 0.5339
[retrain] Epoch 14/30 | Loss: 0.9215 | Accuracy: 0.7183 | Test: 0.5123
[retrain] Epoch

## Compare the original methods and three distance-weighted variants

The default comparison has **five unlearning variants + Original and Retrain**.
Run the method cells independently after Original and Retrain are ready.

- Distance weights: student starts from Original; first wrong prediction along a fixed FGSM direction.
- Distance + Original KL: a separate student starts from Original; retain teacher is Original.
- Distance + Shrink KL: student starts from the distance-weighted model; retain teacher is Boundary Shrink.
  Run both prerequisite method cells first. This student has **20 unlearning epochs** at the default settings;
  the other students have 10. It changes both initialization and teacher, so it is not an isolated teacher ablation.

Distance search uses input normalization (`matched`); the original Shrink keeps `ATTACK_NORMALIZATION`.
On CIFAR-10 the default `repo_exact` baseline therefore also differs in attack normalization.
These are extensions of Shrink, not the additional baseline algorithms in the paper.


In [ ]:
# Incremental execution: this cell defines runners without training anything.
# Run each method cell separately; rerunning a cell retrains only that method.
from boundary_unlearning import attacks, enhanced
importlib.reload(attacks)
importlib.reload(enhanced)
importlib.reload(unlearning)
unlearning.validate_methods(METHODS_TO_RUN, extra_exp=EXTRA_EXP)

run_boundary_shrink = partial(
    unlearning.run_boundary_shrink, device=DEVICE, epochs=UNLEARN_EPOCHS,
    learning_rate=UNLEARN_LR, momentum=MOMENTUM, pin_memory=PIN_MEMORY,
    max_batches_per_epoch=MAX_BATCHES_PER_EPOCH,
    bound=FGSM_BOUND, normalization_mode=ATTACK_NORMALIZATION,
    extra_exp=EXTRA_EXP, curvature_lambda=CURVATURE_LAMBDA,
    reweight_bias=REWEIGHT_BIAS, reweight_slope=REWEIGHT_SLOPE,
    cifar_mean=CIFAR_MEAN, cifar_std=CIFAR_STD,
    attack_mean=REPO_ATTACK_MEAN, attack_std=REPO_ATTACK_STD,
)
run_boundary_expanding = partial(
    unlearning.run_boundary_expanding, device=DEVICE, epochs=UNLEARN_EPOCHS,
    learning_rate=UNLEARN_LR, momentum=MOMENTUM, num_classes=NUM_CLASSES,
    pin_memory=PIN_MEMORY, max_batches_per_epoch=MAX_BATCHES_PER_EPOCH,
)
run_boundary_distance = partial(
    unlearning.run_boundary_distance, device=DEVICE, epochs=UNLEARN_EPOCHS,
    learning_rate=UNLEARN_LR, momentum=MOMENTUM, pin_memory=PIN_MEMORY,
    max_batches_per_epoch=MAX_BATCHES_PER_EPOCH,
    bound=FGSM_BOUND, epsilon_step=WEIGHT_EPSILON_STEP,
    kl_lambda=RETAIN_KL_LAMBDA, temperature=RETAIN_KL_TEMPERATURE,
    cifar_mean=CIFAR_MEAN, cifar_std=CIFAR_STD,
)

unlearned_models = globals().get("unlearned_models", {})
unlearn_stats = globals().get("unlearn_stats", {})

def run_method(method_name):
    if method_name not in METHODS_TO_RUN:
        print(f"Skipped: {method_name}")
        return
    if method_name == unlearning.BOUNDARY_KL_METHOD:
        required = ("boundary_shrink", unlearning.DISTANCE_METHOD)
        missing = [name for name in required if name not in unlearned_models or name not in unlearn_stats]
        if missing:
            raise RuntimeError(f"Run prerequisite method cells first: {missing}")
    method_checkpoint = CHECKPOINT_ROOT / f"{method_name}_{run_tag}.pt"
    method_forget_loader = make_loader(train_forget_set, shuffle=True)
    if method_name == "boundary_shrink":
        model, stats = run_boundary_shrink(original_model, method_forget_loader, method_checkpoint)
    elif method_name == "boundary_expanding":
        model, stats = run_boundary_expanding(original_model, method_forget_loader, method_checkpoint)
    else:
        model, stats = run_boundary_distance(
            original_model, method_forget_loader, method_checkpoint, method=method_name,
            retain_loader=make_loader(train_retain_set, shuffle=True),
            weight_assign_model=unlearned_models.get(unlearning.DISTANCE_METHOD),
            boundary_model=unlearned_models.get("boundary_shrink"),
            dependency_stats=unlearn_stats,
        )
    # A changed prerequisite invalidates its previously distilled dependent model.
    if method_name in {"boundary_shrink", unlearning.DISTANCE_METHOD}:
        unlearned_models.pop(unlearning.BOUNDARY_KL_METHOD, None)
        unlearn_stats.pop(unlearning.BOUNDARY_KL_METHOD, None)
    unlearned_models[method_name] = model
    unlearn_stats[method_name] = stats
    print(f"Ready: {method_name}")


In [ ]:
# Original unlearning baselines. Skip this cell when these models are already ready.
for method_name in ("boundary_shrink", "boundary_expanding"):
    run_method(method_name)


In [ ]:
# Distance weights: initialize from Original
run_method("boundary_shrink_weight_assign")


In [ ]:
# Distance + Original KL: initialize from Original
run_method("boundary_shrink_weight_assign_distill")


In [ ]:
# Distance + Shrink KL: warm start from distance weights
run_method("boundary_shrink_weight_assign_boundary_distill")


In [ ]:
from boundary_unlearning.reporting import results_table, unlearning_details_table

evaluate_model_splits = partial(
    training.evaluate_model_splits, device=DEVICE,
    test_loader=test_loader, test_retain_loader=test_retain_loader,
    test_forget_loader=test_forget_loader,
    train_retain_eval_loader=train_retain_eval_loader,
    train_forget_eval_loader=train_forget_eval_loader,
)

# ============================================================
# PHASE 7: CIFAR-100 FINE-LABEL TEST EVALUATION
# ============================================================

missing_methods = [name for name in METHODS_TO_RUN if name not in unlearned_models]
if missing_methods:
    raise RuntimeError(f"Run the missing method cells before final evaluation: {missing_methods}")

models_to_evaluate = {
    "original": original_model,
    "retrain": retrain_model,
    **{name: unlearned_models[name] for name in METHODS_TO_RUN},
}

rows = []
baseline_train_times = {"original": original_train_seconds, "retrain": retrain_train_seconds}

for name, model in models_to_evaluate.items():
    row = {"model": name}
    row.update(evaluate_model_splits(model, include_train=EVALUATE_TRAIN_SPLITS))

    if name in baseline_train_times:
        row["training_seconds"] = baseline_train_times[name]
    else:
        row["training_seconds"] = None

    row["unlearning_seconds"] = unlearn_stats[name]["seconds"] if name in unlearn_stats else None
    row["fgsm_cross_rate"] = unlearn_stats[name].get("cross_rate") if name in unlearn_stats else None
    row["unlearning_update_steps"] = unlearn_stats[name].get("update_steps") if name in unlearn_stats else None

    stats = unlearn_stats.get(name, {})
    row["total_unlearning_seconds"] = stats.get("total_seconds", stats.get("seconds"))
    row["student_update_steps"] = stats.get("student_update_steps", stats.get("update_steps"))
    row["total_update_steps"] = stats.get("total_update_steps", stats.get("update_steps"))
    row["mean_sample_weight"] = stats.get("mean_sample_weight")
    row["mean_boundary_epsilon"] = stats.get("mean_boundary_epsilon")
    row["initializer"] = stats.get("initializer", "original" if stats else None)
    row["retain_teacher"] = stats.get("retain_teacher")

    rows.append(row)

display(results_table(rows, dataset_name=DATASET_NAME))
display(unlearning_details_table(rows))

metrics_path = RESULT_ROOT / f"{run_tag}_metrics.csv"
with metrics_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

config = {
    "dataset": DATASET_NAME,
    "dataset_slug": DATASET_SLUG,
    "dataset_url": f"https://www.kaggle.com/datasets/{DATASET_SLUG}",
    "num_classes": NUM_CLASSES,
    "label_type": LABEL_TYPE,
    "class_names": CIFAR_CLASSES,
    "input_mean": CIFAR_MEAN,
    "input_std": CIFAR_STD,
    "attack_mean": CIFAR_MEAN if ATTACK_NORMALIZATION == "matched" else REPO_ATTACK_MEAN,
    "attack_std": CIFAR_STD if ATTACK_NORMALIZATION == "matched" else REPO_ATTACK_STD,
    "run_tag_settings": tag_settings,
    "baseline_tag": baseline_tag,
    "baseline_tag_settings": baseline_tag_settings,
    "baseline_checkpoints": {"original": str(original_checkpoint), "retrain": str(retrain_checkpoint)},
    "unlearning_stats": {name: unlearn_stats[name] for name in METHODS_TO_RUN},
    "distance_search": {"epsilon_step": WEIGHT_EPSILON_STEP, "normalization": "matched",
                        "input_mean": CIFAR_MEAN, "input_std": CIFAR_STD},
    "retain_distillation": {"lambda": RETAIN_KL_LAMBDA, "temperature": RETAIN_KL_TEMPERATURE,
                            "batchnorm": "train_on_forget_eval_on_retain"},
    "run_profile": RUN_PROFILE,
    "data_root": str(DATA_ROOT),
    "methods": METHODS_TO_RUN,
    "forget_class": FORGET_CLASS,
    "forget_class_name": CIFAR_CLASSES[FORGET_CLASS],
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "evaluation_dataset": "kaggle_cifar100_test_fine",
    "train_sample_count": len(train_fit_set),
    "test_sample_count": len(evaluation_dataset),
    "extra_exp": EXTRA_EXP,
    "curvature_lambda": CURVATURE_LAMBDA,
    "reweight_bias": REWEIGHT_BIAS,
    "reweight_slope": REWEIGHT_SLOPE,
    "original_epochs": ORIGINAL_EPOCHS,
    "original_learning_rate": ORIGINAL_LR,
    "unlearn_epochs": UNLEARN_EPOCHS,
    "unlearn_learning_rate": UNLEARN_LR,
    "fgsm_bound": FGSM_BOUND,
    "attack_normalization": ATTACK_NORMALIZATION,
    "train_data_shape": list(train_dataset.data.shape),
    "test_data_shape": list(evaluation_dataset.data.shape),
    "device": str(DEVICE),
}

config_path = RESULT_ROOT / f"{run_tag}_config.json"
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

print("\nMetrics saved to:", metrics_path)
print("Configuration saved to:", config_path)
print("Checkpoint folder:", CHECKPOINT_ROOT)


In [ ]:
# Small code update: uncomment these two commands to fetch code from GitHub.
# !git -C /kaggle/working/boundary-unlearning fetch --depth 1 origin main
# !git -C /kaggle/working/boundary-unlearning checkout --detach FETCH_HEAD

import importlib
from boundary_unlearning import attacks, enhanced, unlearning, reporting, artifacts
importlib.invalidate_caches()
for module in (attacks, enhanced, unlearning, reporting, artifacts):
    importlib.reload(module)
print("Modules reloaded. Rerun runner setup, the affected method cell, evaluation, then export.")
print("Existing Original/Retrain models remain in memory; their training cells are unnecessary.")


In [ ]:
# ============================================================
# EXPORT RESULTS: run this cell independently to refresh the ZIP.
# ============================================================
from pathlib import Path
import os
from IPython.display import FileLink, display
from boundary_unlearning.artifacts import export_results
from boundary_unlearning.reporting import results_table, unlearning_details_table

# Use the singular folder name even in a session using the older path settings.
result_dir = Path(OUTPUT_ROOT) / "result"
result_dir.mkdir(parents=True, exist_ok=True)
table_path = result_dir / f"{run_tag}_results.html"
results_table(rows, dataset_name=DATASET_NAME).to_html(table_path)

details_path = result_dir / f"{run_tag}_unlearning_details.html"
unlearning_details_table(rows).to_html(details_path)

archive_path = export_results(
    result_dir,
    checkpoint_dir=CHECKPOINT_ROOT,
    files=[metrics_path, config_path],
)
print(f"Archive: {archive_path} ({archive_path.stat().st_size / (1024 ** 2):.2f} MiB)")
display(FileLink(os.path.relpath(archive_path), result_html_prefix="Download: "))
